# Lesson 2 SQL I | Student Version: Retention Overview and Hypothesis Testing

> Colab version: upload the 5 CSV files directly to `/content/data`.

This notebook uses `pandas + sqlite3` — no DuckDB and no zip extraction needed.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Spotify User Behavior & Conversion Optimization Project: Colab CSV data loading
# ============================================================
# How to use:
# 1. Create a /content/data folder in the Colab file sidebar
# 2. Upload the following 5 CSV files directly to /content/data:
#    users.csv
#    listening_events.csv
#    subscription_events.csv
#    ad_events.csv
#    feature_table.csv
# 3. Run this cell
# ============================================================

import os
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

DATA_DIR = '/content/data'

required_files = {
    'users': 'users.csv',
    'listening_events': 'listening_events.csv',
    'subscription_events': 'subscription_events.csv',
    'ad_events': 'ad_events.csv',
    'feature_table': 'feature_table.csv'
}

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(
        "Cannot find /content/data. Please create /content/data and upload the five CSV files before running this cell."
    )

missing_files = []
for table_name, file_name in required_files.items():
    file_path = os.path.join(DATA_DIR, file_name)
    if not os.path.exists(file_path):
        missing_files.append(file_name)

if missing_files:
    raise FileNotFoundError(
        "Missing files: " + str(missing_files) +
        "\nPlease upload these CSV files directly to /content/data before running this cell."
    )

# Read the CSVs
users_df = pd.read_csv(os.path.join(DATA_DIR, 'users.csv'))
listening_events_df = pd.read_csv(os.path.join(DATA_DIR, 'listening_events.csv'))
subscription_events_df = pd.read_csv(os.path.join(DATA_DIR, 'subscription_events.csv'))
ad_events_df = pd.read_csv(os.path.join(DATA_DIR, 'ad_events.csv'))
feature_table_df = pd.read_csv(os.path.join(DATA_DIR, 'feature_table.csv'))

# Register tables in SQLite; all subsequent SQL queries run against it
conn = sqlite3.connect(':memory:')
users_df.to_sql('users', conn, if_exists='replace', index=False)
listening_events_df.to_sql('listening_events', conn, if_exists='replace', index=False)
subscription_events_df.to_sql('subscription_events', conn, if_exists='replace', index=False)
ad_events_df.to_sql('ad_events', conn, if_exists='replace', index=False)
feature_table_df.to_sql('feature_table', conn, if_exists='replace', index=False)

def sql(query):
    """Run a SQL query against the in-memory SQLite database and return a DataFrame."""
    return pd.read_sql_query(query, conn)

def run_script(query):
    """Run a SQL script, usually for CREATE VIEW / DROP VIEW statements."""
    conn.executescript(query)
    print('SQL script executed successfully.')

print('Data loaded successfully.')
print('Tables available: users, listening_events, subscription_events, ad_events, feature_table')


Data loaded successfully.
Tables available: users, listening_events, subscription_events, ad_events, feature_table


## Suggested Class Pacing

- 0–5 min: load the data and check table granularity.
- 5–45 min: write SQL section by section.
- 45–55 min: interpret results, insights, and actions.
- 55–60 min: recap homework and preview the next lesson.

## Section 0 | Data Granularity Check: What Does One Row Represent in Each Table?

### Question / Hypothesis
Before drawing any business conclusions, we first confirm the granularity of each table. Every JOIN and metric calculation later depends on this step.

### What you need to do
Complete the TODOs in the SQL below.

After writing, run `print(query)` first to check the SQL structure, then run `sql(query)` to see the results.


In [3]:
query = """SELECT 'users' AS table_name,
       COUNT(*) AS row_count,
       COUNT(DISTINCT user_id) AS user_count
FROM users

UNION ALL

SELECT 'listening_events' AS table_name,
       COUNT(*) AS row_count,
       COUNT(DISTINCT user_id) AS user_count
FROM listening_events

UNION ALL

SELECT 'subscription_events' AS table_name,
       COUNT(*) AS row_count,
       COUNT(DISTINCT user_id) AS user_count
FROM subscription_events

UNION ALL

SELECT 'ad_events' AS table_name,
       COUNT(*) AS row_count,
       COUNT(DISTINCT user_id) AS user_count
FROM ad_events

UNION ALL

SELECT 'feature_table' AS table_name,
       COUNT(*) AS row_count,
       COUNT(DISTINCT user_id) AS user_count
FROM feature_table
;"""

sql(query)

,table_name,row_count,user_count
0,users,50000,50000
1,listening_events,1072063,48187
2,subscription_events,104498,34640
3,ad_events,310438,31000
4,feature_table,50000,50000


## Section 1 | Current User Retention Overview: Establish the Business Baseline First

### Question / Hypothesis
Before slicing any user segments, we need the overall picture: what the overall activity, churn, and conversion levels are.

### What you need to do
Complete the TODOs in the SQL below.

After writing, run `print(query)` first to check the SQL structure, then run `sql(query)` to see the results.


In [4]:
query = """SELECT
    COUNT(DISTINCT user_id) AS total_users,
    COUNT(DISTINCT CASE WHEN active_days_30d > 0 THEN user_id END) AS active_users_30d,
    ROUND(AVG(active_days_30d), 2) AS avg_active_days_30d,
    ROUND(AVG(listen_minutes_30d), 2) AS avg_listen_minutes_30d,
    ROUND(AVG(churn_label_14d), 4) AS churn_rate_14d,
    ROUND(AVG(paid_conversion_30d), 4) AS paid_conversion_rate_30d,
    ROUND(AVG(cancel_30d), 4) AS cancel_rate_30d
FROM feature_table
;"""

sql(query)

,total_users,active_users_30d,avg_active_days_30d,avg_listen_minutes_30d,churn_rate_14d,paid_conversion_rate_30d,cancel_rate_30d
0,50000,44118,6.43,38.53,0.4756,0.1739,0.0129


## Section 2 | Split by Subscription Status: Users at Different Lifecycle Stages Should Not Be Analyzed Together

### Question / Hypothesis
Free, trial, and paid users are at different lifecycle stages. A change in overall metrics may just reflect a shift in user mix, not all users actually getting worse.

### What you need to do
Complete the TODOs in the SQL below.

After writing, run `print(query)` first to check the SQL structure, then run `sql(query)` to see the results.


In [5]:
query = """SELECT
    subscription_type,
    COUNT(DISTINCT user_id) AS users,
    ROUND(AVG(active_days_30d), 2) AS avg_active_days_30d,
    ROUND(AVG(listen_minutes_30d), 2) AS avg_listen_minutes_30d,
    ROUND(AVG(churn_label_14d), 4) AS churn_rate_14d,
    ROUND(AVG(paid_conversion_30d), 4) AS paid_conversion_rate_30d,
    ROUND(AVG(cancel_30d), 4) AS cancel_rate_30d
FROM feature_table
GROUP BY subscription_type
ORDER BY users DESC
;"""

sql(query)

,subscription_type,users,avg_active_days_30d,avg_listen_minutes_30d,churn_rate_14d,paid_conversion_rate_30d,cancel_rate_30d
0,free,30006,5.18,29.72,0.5735,0.0010,0.0180
1,individual_premium,14555,8.43,52.46,0.3198,0.4596,0.0000
2,family_premium,3247,8.59,53.34,0.3163,0.4666,0.0000
3,trial,1249,6.30,37.57,0.4652,0.0000,0.0857
4,student_premium,943,8.54,54.04,0.3277,0.4889,0.0000


## Section 3 | Hypothesis 1: Is Low Activity a Churn Risk Signal?

### Question / Hypothesis
For a product like Spotify, the core is usage habit. If a user barely listens to music within 30 days, their subsequent churn risk is most likely higher.

### What you need to do
Complete the TODOs in the SQL below.

After writing, run `print(query)` first to check the SQL structure, then run `sql(query)` to see the results.


In [6]:
query = """WITH user_activity AS (
    SELECT
        user_id,
        active_days_30d,
        listen_minutes_30d,
        paid_conversion_30d,
        churn_label_14d,
        CASE
            WHEN active_days_30d = 0 THEN 'inactive'
            WHEN active_days_30d <= 3 THEN 'low'
            WHEN active_days_30d <= 10 THEN 'medium'
            ELSE 'high'
        END AS active_level
    FROM feature_table
)
SELECT
    active_level,
    COUNT(DISTINCT user_id) AS users,
    ROUND(AVG(active_days_30d), 2) AS avg_active_days_30d,
    ROUND(AVG(listen_minutes_30d), 2) AS avg_listen_minutes_30d,
    ROUND(AVG(churn_label_14d), 4) AS churn_rate_14d,
    ROUND(AVG(paid_conversion_30d), 4) AS paid_conversion_rate_30d
FROM user_activity
GROUP BY active_level
ORDER BY
    CASE active_level
        WHEN 'inactive' THEN 1
        WHEN 'low' THEN 2
        WHEN 'medium' THEN 3
        WHEN 'high' THEN 4
    END
;"""

sql(query)

,active_level,users,avg_active_days_30d,avg_listen_minutes_30d,churn_rate_14d,paid_conversion_rate_30d
0,inactive,5882,0.00,0.00,0.7724,0.1000
1,low,15070,1.90,8.56,0.6663,0.1291
2,medium,18135,6.47,35.28,0.4374,0.1822
3,high,10913,16.10,106.09,0.1156,0.2619


## Section 4 | Hypothesis 2: Does Ad Overload Hurt Retention?

### Question / Hypothesis
Ads monetize free users, but too many ads may hurt the experience. We want to determine whether ad pressure correlates with churn/cancel risk.

### What you need to do
Complete the TODOs in the SQL below.

After writing, run `print(query)` first to check the SQL structure, then run `sql(query)` to see the results.


In [7]:
query = """WITH ad_base AS (
    SELECT
        user_id,
        subscription_type,
        active_level_30d,
        active_days_30d,
        ad_impressions_30d,
        ad_impressions_per_active_day,
        churn_label_14d,
        cancel_30d,
        CASE
            WHEN ad_impressions_per_active_day = 0 THEN 'no_ads'
            WHEN ad_impressions_per_active_day <= 2 THEN 'low'
            WHEN ad_impressions_per_active_day <= 5 THEN 'medium'
            ELSE 'high'
        END AS ad_load_group
    FROM feature_table
    WHERE subscription_type IN ('free', 'trial')
)
SELECT
    ad_load_group,
    COUNT(DISTINCT user_id) AS users,
    ROUND(AVG(active_days_30d), 2) AS avg_active_days_30d,
    ROUND(AVG(ad_impressions_per_active_day), 2) AS avg_ads_per_active_day,
    ROUND(AVG(churn_label_14d), 4) AS churn_rate_14d,
    ROUND(AVG(cancel_30d), 4) AS cancel_rate_30d
FROM ad_base
GROUP BY ad_load_group
ORDER BY
    CASE ad_load_group
        WHEN 'no_ads' THEN 1
        WHEN 'low' THEN 2
        WHEN 'medium' THEN 3
        WHEN 'high' THEN 4
    END
;"""

sql(query)

,ad_load_group,users,avg_active_days_30d,avg_ads_per_active_day,churn_rate_14d,cancel_rate_30d
0,no_ads,7903,0.83,0.00,0.7681,0.0170
1,low,22063,6.92,0.87,0.4928,0.0224
2,medium,1226,3.26,3.09,0.6525,0.0131
3,high,63,1.02,6.56,0.7302,0.0159


## Section 5 | Advanced Control: Compare Ad Load Within the Same Activity Level

### Question / Hypothesis
To reduce confounding from activity level, we compare churn across ad_load_group within the same active_level.

### What you need to do
Complete the TODOs in the SQL below.

After writing, run `print(query)` first to check the SQL structure, then run `sql(query)` to see the results.


In [8]:
query = """WITH ad_base AS (
    SELECT
        user_id,
        active_level_30d,
        ad_impressions_per_active_day,
        churn_label_14d,
        CASE
            WHEN ad_impressions_per_active_day = 0 THEN 'no_ads'
            WHEN ad_impressions_per_active_day <= 2 THEN 'low'
            WHEN ad_impressions_per_active_day <= 5 THEN 'medium'
            ELSE 'high'
        END AS ad_load_group
    FROM feature_table
    WHERE subscription_type IN ('free', 'trial')
)
SELECT
    active_level_30d,
    ad_load_group,
    COUNT(DISTINCT user_id) AS users,
    ROUND(AVG(ad_impressions_per_active_day), 2) AS avg_ads_per_active_day,
    ROUND(AVG(churn_label_14d), 4) AS churn_rate_14d
FROM ad_base
GROUP BY active_level_30d, ad_load_group
HAVING COUNT(DISTINCT user_id) >= 50
ORDER BY active_level_30d, ad_load_group
;"""

sql(query)

,active_level_30d,ad_load_group,users,avg_ads_per_active_day,churn_rate_14d
0,high,low,4656,0.82,0.1622
1,high,medium,92,2.35,0.1413
2,inactive,no_ads,4625,0.00,0.8067
3,low,high,63,6.56,0.7302
4,low,low,6993,1.06,0.6998
5,low,medium,962,3.27,0.7152
6,low,no_ads,2870,0.00,0.7307
7,medium,low,10414,0.77,0.5016
8,medium,medium,172,2.50,0.5756
9,medium,no_ads,399,0.00,0.6040


## Section 6 | Hypothesis 3: Is Trial Exposure Correlated with Paid Conversion?

### Question / Hypothesis
When there are many free users but conversion is low, trial exposure is one of the most common growth levers. We first check whether exposed users show higher paid conversion.

### What you need to do
Complete the TODOs in the SQL below.

After writing, run `print(query)` first to check the SQL structure, then run `sql(query)` to see the results.


In [14]:
query = """
SELECT
    trial_exposed_30d,
    COUNT(DISTINCT user_id) AS users,
    ROUND(AVG(active_days_30d), 2) AS avg_active_days_30d,
    ROUND(AVG(paid_conversion_30d), 4) AS paid_conversion_rate_30d,
    ROUND(AVG(churn_label_14d), 4) AS churn_rate_14d
FROM feature_table
WHERE subscription_type IN ('free', 'trial')
GROUP BY trial_exposed_30d
;"""

# Run after completing the TODOs:
sql(query)

,trial_exposed_30d,users,avg_active_days_30d,paid_conversion_rate_30d,churn_rate_14d
0,0,24052,5.07,0.0006,0.5795
1,1,7203,5.72,0.0021,0.5345


## Section 7 | Subscription Conversion Funnel: Locate Where Users Get Stuck

### Question / Hypothesis
The overall conversion rate only tells us the outcome; the funnel tells us where the break points are.

### What you need to do
Complete the TODOs in the SQL below.

After writing, run `print(query)` first to check the SQL structure, then run `sql(query)` to see the results.


In [16]:
query = """-- TODO: subscription conversion funnel
SELECT
    '1_free_users' AS funnel_step,
    COUNT(DISTINCT user_id) AS users
FROM feature_table
WHERE subscription_type IN ('free', 'trial')

UNION ALL

-- TODO: 2_engaged_free_users: active_days_30d >= 4
SELECT
    '2_engaged_free_users' AS funnel_step,
    COUNT(DISTINCT user_id) AS users
FROM feature_table
WHERE subscription_type IN ('free', 'trial') AND active_days_30d >= 4

UNION ALL

-- TODO: 3_trial_exposed_users: trial_exposed_30d = 1
SELECT
    '3_trial_exposed_users' AS funnel_step,
    COUNT(DISTINCT user_id) AS users
FROM feature_table
WHERE subscription_type IN ('free', 'trial') AND trial_exposed_30d = 1

UNION ALL

-- TODO: 4_trial_started_users: trial_started_30d = 1
SELECT
    '4_trial_started_users' AS funnel_step,
    COUNT(DISTINCT user_id) AS users
FROM feature_table
WHERE subscription_type IN ('free', 'trial') AND trial_started_30d = 1

UNION ALL

-- TODO: 5_paid_converted_users: paid_conversion_30d = 1
SELECT
    '5_paid_converted_users' AS funnel_step,
    COUNT(DISTINCT user_id) AS users
FROM feature_table
WHERE subscription_type IN ('free', 'trial') AND paid_conversion_30d = 1;
"""

# Run after completing the TODOs:
sql(query)

,funnel_step,users
0,1_free_users,31255
1,2_engaged_free_users,15742
2,3_trial_exposed_users,7203
3,4_trial_started_users,2681
4,5_paid_converted_users,30


## Section 8 | Hypothesis 4: Are Mobile Users More Likely to Convert?

### Question / Hypothesis
Device represents usage context and product path. Mobile may be closer to everyday listening, and its conversion path may be smoother.

### What you need to do
Complete the TODOs in the SQL below.

After writing, run `print(query)` first to check the SQL structure, then run `sql(query)` to see the results.


In [19]:
query = """-- TODO: Hypothesis 4: do conversion and churn differ across devices?
SELECT
    primary_device,
    COUNT(DISTINCT user_id) AS users,
    -- TODO: avg_active_days_30d
    ROUND(AVG(active_days_30d), 2) AS avg_active_days_30d,

    -- TODO: trial_exposure_rate_30d
    ROUND(AVG(trial_exposed_30d), 4) AS trial_exposure_rate_30d,

    -- TODO: paid_conversion_rate_30d
    ROUND(AVG(paid_conversion_30d), 4) AS paid_conversion_rate_30d,

    -- TODO: churn_rate_14d
    ROUND(AVG(churn_label_14d), 4) AS churn_rate_14d
FROM feature_table
GROUP BY primary_device
ORDER BY paid_conversion_rate_30d DESC
;"""

# Run after completing the TODOs:
sql(query)

,primary_device,users,avg_active_days_30d,trial_exposure_rate_30d,paid_conversion_rate_30d,churn_rate_14d
0,mobile_ios,19806,6.46,0.2689,0.1911,0.4700
1,mobile_android,17510,6.41,0.2558,0.1716,0.4787
2,smart_speaker,1545,6.78,0.2479,0.1702,0.4563
3,desktop,6548,6.41,0.2434,0.1513,0.4821
4,web,4591,6.34,0.2398,0.1425,0.4851


## Section 9 | Hypothesis 5: Does Content Interaction Improve Stickiness?

### Question / Hypothesis
Long-term retention on a music platform is not just listening time — it also includes whether users build content assets, such as liked songs, playlists, and search preferences.

### What you need to do
Complete the TODOs in the SQL below.

After writing, run `print(query)` first to check the SQL structure, then run `sql(query)` to see the results.


In [20]:
query = """-- TODO: Hypothesis 5: is content interaction correlated with user stickiness?
WITH content_base AS (
    SELECT
        user_id,
        playlist_adds_30d,
        liked_songs_30d,
        search_events_30d,
        churn_label_14d,
        CASE
            -- TODO: if any of playlist / like / search > 0, define as has_content_interaction
            WHEN playlist_adds_30d > 0 OR liked_songs_30d > 0 OR search_events_30d > 0 THEN 'has_content_interaction'
            ELSE 'no_content_interaction'

        END AS content_interaction_group
    FROM feature_table
)
SELECT
    content_interaction_group,
    COUNT(DISTINCT user_id) AS users,
    -- TODO: avg_playlist_adds_30d
    ROUND(AVG(playlist_adds_30d), 2) AS avg_playlist_adds_30d,

    -- TODO: avg_liked_songs_30d
    ROUND(AVG(liked_songs_30d), 2) AS avg_liked_songs_30d,

    -- TODO: avg_search_events_30d
    ROUND(AVG(search_events_30d), 2) AS avg_search_events_30d,

    -- TODO: churn_rate_14d
    ROUND(AVG(churn_label_14d), 4) AS churn_rate_14d
FROM content_base
GROUP BY content_interaction_group
;"""

# Run after completing the TODOs:
sql(query)

,content_interaction_group,users,avg_playlist_adds_30d,avg_liked_songs_30d,avg_search_events_30d,churn_rate_14d
0,has_content_interaction,33720,0.72,1.03,2.61,0.3696
1,no_content_interaction,16280,0.00,0.00,0.00,0.6950


## Homework Submission Requirements

1. Submit the completed notebook.
2. Write one insight sentence after each section.
3. Write one business action after each insight.
4. Note at least one limitation: e.g., correlation is not causation, sample composition differences, future information leakage.